<a href="https://colab.research.google.com/github/EMej34/das172-examen2-Edwin-Reyes./blob/main/Script_principal_de_ejecuci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
from aerocargo_matrix import (
    validar_matrices,
    calcular_ocupacion,
    evaluar_balance,
    extraer_submatriz_critica,
)


def imprimir_encabezado_modulo(numero, nombre):
    print("\n" + "-" * 72)
    print(f"MODULO {numero}: {nombre}")
    print("-" * 72)


def imprimir_matriz(matriz, titulo, decimales=1):
    print(f"\n{titulo}")
    for fila in matriz:
        valores = "  ".join(f"{valor:8.{decimales}f}" for valor in fila)
        print(f"  [ {valores} ]")


def imprimir_tabla_ocupacion(matriz_porcentajes, celdas_sobrecargadas):
    print("\nDetalle de ocupacion por celda:")
    print(f"  {'Fila':<6}{'Columna':<10}{'Ocupacion':<14}{'Estado'}")
    print("  " + "-" * 40)
    celdas_criticas = set(celdas_sobrecargadas)
    filas = len(matriz_porcentajes)
    columnas = len(matriz_porcentajes[0])
    for i in range(filas):
        for j in range(columnas):
            porcentaje = matriz_porcentajes[i][j]
            estado = "SOBRECARGA" if (i, j) in celdas_criticas else "normal"
            print(f"  {i:<6}{j:<10}{porcentaje:<14.1f}{estado}")


def main():
    # ---------------------------------------------------------------
    # DATOS DE PRUEBA: bahia de carga de 4 filas (proa -> popa) x
    # 5 columnas (izquierda -> derecha), tomados como ejemplo para
    # demostrar el flujo completo del sistema.
    # ---------------------------------------------------------------
    cargas_reales = [
        [420, 380, 300, 410, 390],
        [500, 470, 260, 480, 520],
        [310, 300, 295, 305, 300],
        [610, 400, 310, 420, 600],
    ]

    capacidades_maximas = [
        [450, 450, 450, 450, 450],
        [450, 450, 450, 450, 450],
        [450, 450, 450, 450, 450],
        [450, 450, 450, 450, 450],
    ]

    tolerancia_kg = 100.0
    ventana_k, ventana_p = 2, 2

    print("=" * 72)
    print(" AeroCargo-Matrix - Auditoria y Balance de Carga en Bahia")
    print("=" * 72)

    print("\nDatos de prueba:")
    imprimir_matriz(cargas_reales, "Matriz de Cargas Reales (kg)", decimales=0)
    imprimir_matriz(capacidades_maximas, "Matriz de Capacidades Maximas (kg)", decimales=0)

    # MODULO 1: Validacion y Coherencia Dimensional
    imprimir_encabezado_modulo(1, "Validacion y Coherencia Dimensional")
    es_valida = validar_matrices(cargas_reales, capacidades_maximas)
    print(f"Matrices validas: {es_valida}")

    if not es_valida:
        print("Las matrices de prueba no son validas. Abortando ejecucion.")
        return

    # MODULO 2: Calculo de Ocupacion y Deteccion de Sobrecarga
    imprimir_encabezado_modulo(2, "Calculo de Ocupacion y Deteccion de Sobrecarga")
    resultado_ocupacion = calcular_ocupacion(cargas_reales, capacidades_maximas)
    imprimir_matriz(resultado_ocupacion["matriz_porcentajes"],
                     "Matriz de Porcentajes de Ocupacion (%)")
    imprimir_tabla_ocupacion(resultado_ocupacion["matriz_porcentajes"],
                              resultado_ocupacion["celdas_sobrecargadas"])

    total_celdas = len(cargas_reales) * len(cargas_reales[0])
    total_sobrecargadas = len(resultado_ocupacion["celdas_sobrecargadas"])
    print(f"\nResumen: {total_sobrecargadas} de {total_celdas} celdas en sobrecarga.")

    # MODULO 3: Evaluacion de Balance y Simetria
    imprimir_encabezado_modulo(3, "Evaluacion de Balance y Simetria")
    resultado_balance = evaluar_balance(cargas_reales, tolerancia_kg)
    print(f"{'Fila':<8}{'Peso total (kg)'}")
    print("-" * 26)
    for i, peso in enumerate(resultado_balance["pesos_por_fila"]):
        print(f"{i:<8}{peso:.1f}")
    print(f"\nDesbalance lateral: {resultado_balance['desbalance_lateral']:.1f} kg "
          f"(tolerancia: {tolerancia_kg:.1f} kg)")
    print(f"Balance aprobado: {resultado_balance['balance_aprobado']}")

    # MODULO 4: Extraccion de Submatriz de Sobrecarga Critica
    imprimir_encabezado_modulo(4, "Extraccion de Submatriz de Sobrecarga Critica")
    submatriz_critica = extraer_submatriz_critica(
        resultado_ocupacion["matriz_porcentajes"], ventana_k, ventana_p
    )
    if submatriz_critica is not None:
        imprimir_matriz(submatriz_critica,
                         f"Submatriz critica ({ventana_k}x{ventana_p}) de mayor ocupacion")
    else:
        print(f"No se pudo extraer una submatriz de tamano {ventana_k}x{ventana_p}.")

    print("\n" + "=" * 72)


if __name__ == "__main__":
    main()